# Inter-channel displacement for moving objects


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 8)


In [ ]:
# Input files (same folder as notebook)
base = Path('.')
paths = {
    'bid1': base / 'tile-bid1.png',
    'bid2': base / 'tile-bid2.png',
    'bid3': base / 'tile-bid3.png',
    'bid4': base / 'tile-bid4.png',
}

missing = [name for name, p in paths.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing input files: {missing}')

bands = {
    name: cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    for name, path in paths.items()
}

if any(v is None for v in bands.values()):
    bad = [k for k, v in bands.items() if v is None]
    raise ValueError(f'Failed to read: {bad}')

shapes = {arr.shape for arr in bands.values()}
if len(shapes) != 1:
    raise ValueError(f'Bands have mismatched shapes: {shapes}')

print('Loaded bands:', {k: v.shape for k, v in bands.items()})


In [ ]:
def stretch_to_uint8(img, p_low=2, p_high=98):
    # img = img.astype(np.float32)
    # lo, hi = np.percentile(img, (p_low, p_high))
    # if hi <= lo:
    #     return np.zeros_like(img, dtype=np.uint8)
    # scaled = np.clip((img - lo) / (hi - lo), 0, 1)
    # return (scaled * 255).astype(np.uint8)
    return img

# Stretch each band for visualization and robust thresholding
bid1 = stretch_to_uint8(bands['bid1'])
bid2 = stretch_to_uint8(bands['bid2'])
bid3 = stretch_to_uint8(bands['bid3'])
bid4 = stretch_to_uint8(bands['bid4'])

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for i, (name, img) in enumerate([('bid1 (Red)', bid1), ('bid2 (Green)', bid2), ('bid3 (Blue)', bid3), ('bid4 (NIR)', bid4)]):
    ax[i].imshow(img, cmap='gray')
    ax[i].set_title(name)
    ax[i].axis('off')
plt.tight_layout()


In [ ]:
# Build true-color preview from Landsat 5/7 mapping
# RGB = (bid3, bid2, bid1)
rgb = np.dstack([bid1, bid2, bid3])

plt.figure(figsize=(7, 7))
plt.imshow(rgb)
plt.title('True-color composite (R=bid1, G=bid2, B=bid3)')
plt.axis('off')


In [ ]:
# Stage 2 - Inter-channel displacement map
r, g, b = bid1, bid2, bid3

rg = cv2.subtract(bid2, bid1)
gb = cv2.subtract(bid2, bid3)
rb = cv2.subtract(bid1, bid4)

motion_f = rg.astype(np.float32) + gb.astype(np.float32) + rb.astype(np.float32)
motion = cv2.normalize(motion_f, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for i, (title, img) in enumerate([
    ('|R-G|', rg),
    ('|G-B|', gb),
    ('|R-B|', rb),
    ('motion', motion),
]):
    ax[i].imshow(img, cmap='gray')
    ax[i].set_title(title)
    ax[i].axis('off')
plt.tight_layout()


In [ ]:
motion = cv2.GaussianBlur(motion, (5, 5), 0)
# Stage 3 - Threshold
threshold_value = 60
_, thresh = cv2.threshold(motion, threshold_value, 255, cv2.THRESH_BINARY)

plt.figure(figsize=(7, 7))
plt.imshow(thresh, cmap='gray')
plt.title(f'Thresholded motion (>{threshold_value})')
plt.axis('off')


In [ ]:
# Stage 4 - Morphological cleanup
kernel = np.ones((3, 3), np.uint8)
clean = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
clean = cv2.dilate(clean, kernel, iterations=3)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(thresh, cmap='gray')
ax[0].set_title('Before cleanup')
ax[0].axis('off')
ax[1].imshow(clean, cmap='gray')
ax[1].set_title('After open + dilate')
ax[1].axis('off')
plt.tight_layout()


In [ ]:
# Stage 5 - Connected component detection
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(clean, connectivity=8)

# stats columns: [x, y, width, height, area]
min_area = 8
max_area = 2000

candidates = []
for i in range(1, num_labels):  # skip background label 0
    x, y, w, h, area = stats[i]
    if min_area <= area <= max_area:
        cx, cy = centroids[i]
        candidates.append({
            'label': i,
            'x': int(x), 'y': int(y), 'w': int(w), 'h': int(h),
            'area': int(area),
            'cx': float(cx), 'cy': float(cy),
        })

print(f'Total labels (excluding background): {num_labels - 1}')
print(f'Candidates after area filter [{min_area}, {max_area}]: {len(candidates)}')
if candidates:
    print('First 10 candidates:')
    for c in candidates[:10]:
        print(c)


In [ ]:
# Overlay candidate boxes on true-color image
overlay = rgb.copy()
for c in candidates:
    x, y, w, h = c['x'], c['y'], c['w'], c['h']
    cv2.rectangle(overlay, (x, y), (x + w, y + h), color=(255, 255, 0), thickness=1)

plt.figure(figsize=(8, 8))
plt.imshow(overlay)
plt.title(f'Candidate moving objects: {len(candidates)}')
plt.axis('off')
